In [2]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from firecrawl import FirecrawlApp
from pprint import pprint
from groq import Groq
import json
from tiktoken import encoding_for_model
from urllib.parse import urlparse
import datetime
from together import Together
from openai import OpenAI

load_dotenv()

False

In [3]:
#client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
scraper = FirecrawlApp(api_key=os.getenv("FC_API_KEY"))
urls = [
    "https://www.kotsovolos.gr/mobile-phones-gps/mobile-phones/smartphones/312002-smartphone-iphone-16-2",
    "https://www.plaisio.gr/tilefonia-tablet/tilefona/smartphones/apple-iphone-16-128gb-black_4621573",
    "https://www.public.gr/product/tilefonia/kinita-smartphones/apple-iphone-16-128gb--black/1966492",
    "https://www.istorm.gr/iphone-16-128gb-black.html"
]
parameters = {'formats': ['markdown'], 'excludeTags': ['a', 'img', 'href', 'script', 'style']}



In [4]:
system_prompt = """You are a proffesional data retriever. You are provided with markdown scraped content from e-commerce sites.
    You are tasked with extracting the price of the product and the availability of the product.
    The price must be in the format of a number with no currency symbol and two decimal places with a comma as a decimal separator.
    If the base domain of the url is 'skroutz.gr' AND multiple prices are available, you must extract the lowest price.
    Availability is either "In stock" or "Out of stock".
    You are also tasked with extracting the product name and the brand.
    You are also tasked with extracting the product SKU Code. 
    You are also tasked with extracting the secret code of the product. The secret code comes just after the SKU Code. If it isn't available, return ''
    You provide your output in JSON format with no markdown or other formatting and definately no '\n' or '\t' or '\r':
    {
        "price": "1000,00",
        "availability": "In stock",
        "product_name": "iPhone 16",
        "brand": "Apple",
        "sku_code": "1234567890"
    }
    """

In [22]:
system_prompt = """You are a proffesional data retriever. You are provided with markdown scraped content from e-commerce sites.
    You are tasked with extracting the price of the product and the availability of the product.
    The price must be in the format of a number with no currency symbol and two decimal places with a comma as a decimal separator.
    Availability is either "In stock" or "Out of stock".
    You are also tasked with extracting the product name and the brand.
    You are also tasked with extracting the product SKU Code. 
    You provide your output in JSON format with no markdown or other formatting and definately no '\n' or '\t' or '\r':
    {
        "price": "1000,00",
        "availability": "In stock",
        "product_name": "iPhone 16",
        "brand": "Apple",
        "sku_code": "1234567890"
    }
    """

In [5]:
pprint(raw_data['markdown'])

('[iframe](javascript:void(0))\n'
 '\n'
 '9 εικόνες\n'
 '\n'
 '![](https://assets.kotsovolos.gr/product/312002-b.jpg)\n'
 '\n'
 '![next-img](https://assets.kotsovolos.gr/product/312002-s.jpg)\n'
 '\n'
 '![next-img](https://assets.kotsovolos.gr/product/312002-1-s.jpg)\n'
 '\n'
 '![next-img](https://assets.kotsovolos.gr/product/312002-2-s.jpg)\n'
 '\n'
 '![next-img](https://assets.kotsovolos.gr/product/312002-3-s.jpg)\n'
 '\n'
 'Μέγεθος Οθόνης σε Ίντσες:\n'
 '\n'
 '6.1\n'
 '\n'
 'Κύρια Κάμερα:\n'
 '\n'
 'Dual 48 Mp & 13 Mp\n'
 '\n'
 'Selfie Κάμερα:\n'
 '\n'
 '12 Mp\n'
 '\n'
 'Πλήθος Πυρήνων:\n'
 '\n'
 'Hexa Core\n'
 '\n'
 'Χωρητικότητα σε GB:\n'
 '\n'
 '128\n'
 '\n'
 'Υποστηρικτική Τεχνολογία:\n'
 '\n'
 'Κινητικές αναπηρίες, Μειωμένη όραση, Μειωμένη ακοή\n'
 '\n'
 '![next-img](/images/image_192.svg)\n'
 '\n'
 '- 312002\n'
 '\n'
 '## Apple iPhone 16 128GB White Κινητό Smartphone\n'
 '\n'
 '31200274539001https://www.kotsovolos.gr/mobile-phones-gps/mobile-phones/smartphones/312002-smartphon

In [27]:
class DataExtractor:
    def __init__(self):
        self.parameters = parameters
        #self.client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
        self.client = Groq(api_key=os.getenv("GROQ_API_KEY"))
        self.model = "llama-3.1-8b-instant"
        self.scraper = scraper
        self.tokenizer = encoding_for_model('gpt-4o-mini')

    def token_calc(self, messages):
        total_input_tokens = 0
        for message in messages:
            total_input_tokens += len(self.tokenizer.encode(message["content"]))
        return total_input_tokens

    def extract_data(self, url):
        markd = self.scraper.scrape_url(url, params=self.parameters)
        raw_data = ' '.join(markd['markdown'].split()[:500])
        print(raw_data)
        print(system_prompt)
        messages = [
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user", 
                "content": raw_data
            }
        ]

        input_system_tokens = self.token_calc([{"content": system_prompt}])
        input_user_tokens   = self.token_calc([{"content": raw_data}])

        chat_completion = self.client.chat.completions.create(
            messages=messages,
            model=self.model,
            response_format={"type": "json_object"},
        )

        response_content = chat_completion.choices[0].message.content
        output_tokens = len(self.tokenizer.encode(response_content))

        print(f"Input system tokens: {input_system_tokens}")
        print(f"Input user tokens: {input_user_tokens}")
        print(f"Output tokens: {output_tokens}")
        #print(response_content)

        data = json.loads(response_content)
        current_time = datetime.datetime.now()
        data['timestamp'] = current_time.strftime("%Y-%m-%d %H:%M:%S")

        # Extract domain from raw_data['url'] and add to data dictionary
        parsed_url = urlparse(url)
        data['domain'] = parsed_url.netloc.split('.')[-2] + '.' + parsed_url.netloc.split('.')[-1]
        return data, input_system_tokens, input_user_tokens, output_tokens





In [28]:
tracker = DataExtractor()
data, input_system_tokens, input_user_tokens, output_tokens = tracker.extract_data(urls[3])
pprint(data) 
print(f"\nTotal tokens used: {input_system_tokens + input_user_tokens + output_tokens}")

Search # iPhone 16 128GB Black - Προτεινόμενα αξεσουάρ - Τεχνικά χαρακτηριστικά - Σύγκριση - Περιγραφή - Περιγραφή - Σύγκριση - Τεχνικά χαρακτηριστικά - Προτεινόμενα αξεσουάρ ### iPhone 16 Plus 128GB Black Αναμένεται σύντομα IPhone 16. Εισάγει τον Έλεγχο Κάμερας. Κάμερα Fusion 48MP. Πέντε έντονα χρώματα. Και το A18 chip. ## Αλλαγή χρώματος ## Αλλαγή μεγέθους ## Αλλαγή χωρητικότητας ## Αλλαγή μεγέθους Band ## Αλλαγή χρώματος Band ## Επιλογή γυαλιού _Παράδοση:_ _Διαθέσιμο_ _**Express Delivery** σε 4 ώρεςΜάθε περισσότερα_ _Αποστολή_ _στο χώρο σου_ _Παραλαβή από κατάστημα: Δες διαθεσιμότητα_ X # **Express Delivery** Ισχύει μόνο για περιοχές εντός λεκανοπεδίου Αττικής και εντός αστικού κέντρου Θεσσαλονίκης, εφόσον το προϊόν είναι αμεσα διαθέσιμο, **μόνο με 6€**. Οι παραδόσεις ξεκινάνε στις 10:00 από Δευτέρα έως Παρασκευή. Ισχύει για παραγγελίες που καταχωρούνται μέχρι τις 16:00. Οι παραγγελίες που καταχωρούνται μετά τις 16:00 ή ΣΚ ή αργία, παραδίδονται την επόμενη εργάσιμη ημέρα. Συνέχεια X

In [23]:
tracker = DataExtractor()
for i in urls:
    data, input_system_tokens, input_user_tokens, output_tokens = tracker.extract_data(i)
    pprint(data)
    print(f"\nTotal tokens used: {input_system_tokens + input_user_tokens + output_tokens}")


9 εικόνες Μέγεθος Οθόνης σε Ίντσες: 6.1 Κύρια Κάμερα: Dual 48 Mp & 13 Mp Selfie Κάμερα: 12 Mp Πλήθος Πυρήνων: Hexa Core Χωρητικότητα σε GB: 128 Υποστηρικτική Τεχνολογία: Κινητικές αναπηρίες, Μειωμένη όραση, Μειωμένη ακοή - 312002 - 979.00 ## Apple iPhone 16 128GB White Κινητό Smartphone 31200274539001https://www.kotsovolos.gr/mobile-phones-gps/mobile-phones/smartphones/312002-smartphone-iphone-16-2 5 (2) Αξιολογήσεις (29) iPhone 16. Παρουσιάζουμε τον Έλεγχο Κάμερας. Κάμερα Fusion 48 MP. Πέντε ζωντανά χρώματα. Και το Α18 chip. Δες την αναλυτική περιγραφή Άμεσα διαθέσιμο €979.00 € Χωρητικότητα:128GB Απόκτησέ το Απαλλαγή ΦΠΑ 24% για επιχειρήσειςΜάθε εδώ αν τη δικαιούσαι Περισσότερα Oλοκληρωμένες υπηρεσίες Total Support Insurance Premium Smartphone Κάλυψη για το smartphone σου συνολικής διάρκειας 2 ετών με κάλυψη από κλοπή ή ληστεία παγκοσμίως για το 1ο έτος του συμβολαίου και
You are a proffesional data retriever. You are provided with markdown scraped content from e-commerce sites.
    Y

In [57]:
pprint(data)

{'availability': 'In stock',
 'brand': 'Apple',
 'domain': 'kotsovolos.gr',
 'price': '979',
 'product_name': 'iPhone 16 128GB White',
 'secret_code': '74539001',
 'sku_code': '312002',
 'timestamp': '2025-01-06 15:07:15'}


In [65]:
data1 = extract_data(url1)
data2 = extract_data(url3)
data3 = extract_data(url4)

pprint(data) 

{
    "price": "979.00",
    "availability": "In stock",
    "product_name": "Apple iPhone 16 128GB Black",
    "brand": "Apple",
    "sku_code": "4621573",
    "secret_code": ""
}
{
    "price": "979.00",
    "availability": "In stock",
    "product_name": "Apple iPhone 16 128GB - Black",
    "brand": "Apple",
    "sku_code": "1966492",
    "secret_code": ""
}
{
    "price": "979.00",
    "availability": "In stock",
    "product_name": "iPhone 16 128GB Black",
    "brand": "Apple",
    "sku_code": "MYE73QL/A",
    "secret_code": "746020"
}
{'availability': 'In stock',
 'brand': 'Apple',
 'domain': 'kotsovolos.gr',
 'price': '979.00',
 'product_name': 'Apple iPhone 16 128GB White',
 'secret_code': '',
 'sku_code': '31200274539001',
 'timestamp': '2025-01-06 16:18:32'}


In [66]:
pprint(data)
pprint(data1)
pprint(data2)
pprint(data3)


{'availability': 'In stock',
 'brand': 'Apple',
 'domain': 'kotsovolos.gr',
 'price': '979.00',
 'product_name': 'Apple iPhone 16 128GB White',
 'secret_code': '',
 'sku_code': '31200274539001',
 'timestamp': '2025-01-06 16:18:32'}
{'availability': 'In stock',
 'brand': 'Apple',
 'domain': 'plaisio.gr',
 'price': '979.00',
 'product_name': 'Apple iPhone 16 128GB Black',
 'secret_code': '',
 'sku_code': '4621573',
 'timestamp': '2025-01-06 16:19:48'}
{'availability': 'In stock',
 'brand': 'Apple',
 'domain': 'public.gr',
 'price': '979.00',
 'product_name': 'Apple iPhone 16 128GB - Black',
 'secret_code': '',
 'sku_code': '1966492',
 'timestamp': '2025-01-06 16:19:58'}
{'availability': 'In stock',
 'brand': 'Apple',
 'domain': 'istorm.gr',
 'price': '979.00',
 'product_name': 'iPhone 16 128GB Black',
 'secret_code': '746020',
 'sku_code': 'MYE73QL/A',
 'timestamp': '2025-01-06 16:20:09'}


In [67]:
data4 = extract_data('https://www.mistore-greece.gr/Products/Smartphones/POCO/smartphone-poco-c65-black-6-128gb.aspx')
pprint(data4)


{
    "price": "139.90",
    "availability": "In stock",
    "product_name": "Smartphone POCO C65 6/128GB Black",
    "brand": "Xiaomi",
    "sku_code": "709680",
    "secret_code": ""
}
{'availability': 'In stock',
 'brand': 'Xiaomi',
 'domain': 'mistore-greece.gr',
 'price': '139.90',
 'product_name': 'Smartphone POCO C65 6/128GB Black',
 'secret_code': '',
 'sku_code': '709680',
 'timestamp': '2025-01-06 16:32:57'}


In [34]:
# For Groq's llama model, we'll use the llama tokenizer
# Note: This is an approximation since the exact tokenizer may vary
tokenizer = encoding_for_model("gpt-3.5-turbo")  # Using this as a close approximation

# Calculate tokens for system prompt
system_tokens = len(tokenizer.encode(system_prompt))

# Calculate tokens for user input (markdown from URL)
user_tokens = len(tokenizer.encode(raw_data['markdown']))  # Changed from data to raw_data

# Calculate tokens in the response
response_tokens = len(tokenizer.encode(str(data.get('price', ''))))  # Added str() and .get() with default

print(f"\nToken Usage Analysis:")
print(f"Tokens sent:")
print(f"  - System prompt: {system_tokens}")
print(f"  - User input: {user_tokens}")
print(f"  - Total sent: {system_tokens + user_tokens}")
print(f"\nTokens received:")
print(f"  - Response: {response_tokens}")
print(f"\nTotal tokens in conversation: {system_tokens + user_tokens + response_tokens}")



Token Usage Analysis:
Tokens sent:
  - System prompt: 169
  - User input: 2686
  - Total sent: 2855

Tokens received:
  - Response: 1

Total tokens in conversation: 2856
